# A3: Self-Supervised Learning

In this lab, we will explore **Self-Supervised Learning (SSL)** — learning powerful visual representations **without any human labels**.

---

## Background: Why Self-Supervised Learning?

### The Labeling Bottleneck

Supervised deep learning requires massive labeled datasets. ImageNet has **1.2 million** labeled images that took years and thousands of human annotators to produce. Medical imaging datasets may have only hundreds of labeled examples. In many real-world domains, unlabeled data is abundant but labels are expensive, slow, or require expert knowledge.

SSL solves this by using the **structure of the data itself** as the supervision signal:

```
Raw Unlabeled Data → Self-Supervised Pretraining → Encoder with good representations
                                                          ↓
                                              Fine-tune with few labels → High accuracy
```

This is the **pre-train → fine-tune** paradigm that dominates modern deep learning.

---

## Pretext Tasks

A **pretext task** is an artificial task constructed from unlabeled data that forces the model to learn useful representations as a side effect.

### Early Pretext Tasks (2014–2019)

| Pretext Task | Idea | What it learns |
|---|---|---|
| **Rotation prediction** | Predict 0°/90°/180°/270° rotation | Upright object structure |
| **Jigsaw puzzle** | Predict permutation of shuffled patches | Spatial relationships |
| **Colorization** | Predict color from grayscale | Object/material semantics |
| **Inpainting** | Reconstruct masked regions | Local texture + global context |
| **Relative patch location** | Predict position of one patch relative to another | Spatial reasoning |

These methods worked but were limited — the pretext task didn't perfectly align with downstream tasks.

### Modern SSL: Instance Discrimination (2020–)

The breakthrough came from reframing SSL as **instance discrimination**: treat each image as its own class. Two augmented views of the same image should have similar representations; views from different images should be different.

```
x ──[aug₁]──▶ view₁ ──▶ f(·) ──▶ z₁ ─┐
 └──[aug₂]──▶ view₂ ──▶ f(·) ──▶ z₂ ─┴──▶ z₁ and z₂ should be close
                                             z₁ and z₃ (diff image) should be far
```

This is **augmentation-invariant** representation learning — the encoder must capture what's consistent across augmentations (semantic content) and ignore what changes (color jitter, crop position).

---

## Data Augmentation is the Key

The choice of augmentation determines what the model learns to be invariant to:

| Augmentation | Invariance learned |
|---|---|
| RandomCrop | Position, scale |
| ColorJitter | Color, brightness, contrast |
| Grayscale | Color entirely |
| GaussianBlur | High-frequency texture |
| HorizontalFlip | Left-right orientation |

SimCLR showed that **color jitter + random crop** together are the most critical augmentations for learning good visual representations.

---

## Two Schools: Contrastive vs Non-Contrastive

### Contrastive Methods (need negatives)

Pull positive pairs together, push negative pairs apart in feature space.

$$\mathcal{L}_{contrastive} = -\log \frac{\exp(\text{sim}(z_i, z_j)/\tau)}{\sum_{k \neq i} \exp(\text{sim}(z_i, z_k)/\tau)}$$

**Problem**: need many negatives → huge batch sizes (SimCLR uses 4096+). Memory-intensive.

### Non-Contrastive Methods (no negatives needed)

Learn without explicit negative pairs. Prevent collapse through architecture or regularization tricks.

**Problem**: Without negatives, what stops the model from outputting the same vector for everything? (representational collapse)

| Method | Collapse prevention |
|---|---|
| **BYOL** | Asymmetric predictor MLP + EMA teacher |
| **DINO** | Centering trick + EMA teacher |
| **Barlow Twins** | Cross-correlation matrix regularization |
| **VICReg** | Variance + invariance + covariance loss |

---

## The SSL Timeline (2020–2022)

| Model | Year | Key Idea | Limitation |
|---|---|---|---|
| **SimCLR** | 2020 | Contrastive: push same-image crops together | Needs huge batch (4096+) for negatives |
| **MoCo v2** | 2020 | Memory bank of negatives — no large batch needed | More complex training setup |
| **BYOL** | 2020 | No negatives — teacher-student with EMA | Asymmetric predictor MLP needed to avoid collapse |
| **SwAV** | 2020 | Cluster assignments as pseudo-labels | Requires online clustering |
| **DINO** | 2021 | Self-distillation on ViT + centering trick | More complex, stunning results |
| **MAE** | 2022 | Reconstruct masked patches — simpler, scalable | No contrastive signal; needs longer training |

**This lab focuses on SimCLR (the contrastive foundation) and DINO (the self-distillation highlight).** We end by visualizing DINO's famous attention maps — the result that shows the model learns object segmentation *without ever training on segmentation labels*.

---

## What Makes a Good Representation?

After pre-training, we evaluate representations with **linear evaluation**: freeze the encoder, train only a single linear layer on top with labels. If a simple linear classifier achieves high accuracy, the encoder has learned linearly separable, semantically meaningful features.

```
Pretrained Encoder (frozen) → h → Linear classifier → class label
                                        ↑ only this is trained with labels
```

Good SSL representations:
- **Cluster** — same-class images close in feature space
- **Separate** — different-class images far apart
- **Transfer** — useful across many downstream tasks
- **Label-efficient** — achieve high accuracy with few labeled examples

---

**Papers:** [SimCLR](https://arxiv.org/abs/2002.05709) · [DINO](https://arxiv.org/abs/2104.14294) · [MAE](https://arxiv.org/abs/2111.06377) · [ViT](https://arxiv.org/abs/2010.11929)

**Code credits:** DINO adapted from [facebookresearch/dino](https://github.com/facebookresearch/dino) (Apache 2.0) · NT-Xent from [sthalles/SimCLR](https://github.com/sthalles/SimCLR) (MIT) · ViT backbone via [timm](https://github.com/huggingface/pytorch-image-models)

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import random, os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
os.makedirs('saved', exist_ok=True)

---
## Part 1: SimCLR — The Contrastive Baseline

SimCLR (Chen et al., 2020) is the foundation of modern SSL. It answers:

> *"Two crops of the same image should be close in feature space. Crops from different images should be far apart."*

### Architecture
```
Image x ──[aug]──▶ x_i ──[Encoder f]──▶ h_i ──[Projector g]──▶ z_i ──┐
        └──[aug]──▶ x_j ──[Encoder f]──▶ h_j ──[Projector g]──▶ z_j ──┴──▶ NT-Xent Loss
```

**Why a projector head?** The loss is computed on `z`, but `h` is used for downstream tasks. The projector absorbs augmentation-specific information so the encoder doesn't have to — counterintuitively, removing the projector at evaluation time gives *better* features.

**Temperature τ = 0.5:** Controls how sharp the contrastive distribution is. Low τ → model is penalized heavily for confusing any negative. High τ → softer, easier task.

### Let's set up

<img src="figures/simclr_arch.png" width="750"/>

*SimCLR: two independently augmented views of the same image share an encoder and projector, then NT-Xent loss pulls positive pairs together and pushes all other pairs apart.*

In [ ]:
class SimCLRAugmentation:
    """Returns two independently augmented views of the same image."""
    def __init__(self, image_size=32):
        self.transform = transforms.Compose([
            transforms.RandomResizedCrop(image_size),
            transforms.RandomHorizontalFlip(),
            transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            transforms.RandomGrayscale(p=0.2),
            transforms.GaussianBlur(kernel_size=3),
            transforms.ToTensor(),
            transforms.Normalize([0.4914, 0.4822, 0.4465], [0.2023, 0.1994, 0.2010])
        ])
    def __call__(self, x):
        return self.transform(x), self.transform(x)


class CIFAR10SSL(Dataset):
    def __init__(self, root='./data', train=True):
        self.dataset = torchvision.datasets.CIFAR10(root=root, train=train, download=True)
        self.augment = SimCLRAugmentation()
    def __len__(self): return len(self.dataset)
    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        x_i, x_j = self.augment(img)
        return x_i, x_j, label


class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature
    def forward(self, z_i, z_j):
        N = z_i.shape[0]
        z_i = F.normalize(z_i, dim=1)
        z_j = F.normalize(z_j, dim=1)
        z = torch.cat([z_i, z_j], dim=0)
        sim = torch.mm(z, z.T) / self.temperature
        mask = torch.eye(2 * N, dtype=torch.bool, device=z.device)
        sim = sim.masked_fill(mask, float('-inf'))
        labels = torch.cat([torch.arange(N, 2*N), torch.arange(0, N)]).to(z.device)
        return F.cross_entropy(sim, labels)


class SimCLR(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = torchvision.models.resnet18(weights=None)
        resnet.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        resnet.maxpool = nn.Identity()
        self.encoder = nn.Sequential(*list(resnet.children())[:-1])
        self.projector = nn.Sequential(
            nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, 128)
        )
    def forward(self, x_i, x_j):
        h_i = torch.flatten(self.encoder(x_i), 1)
        h_j = torch.flatten(self.encoder(x_j), 1)
        return self.projector(h_i), self.projector(h_j), h_i, h_j

In [ ]:
# --- Train SimCLR ---
BATCH_SIZE, EPOCHS = 256, 10
train_loader = DataLoader(CIFAR10SSL(), batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, drop_last=True)
simclr    = SimCLR().to(device)
criterion = NTXentLoss(temperature=0.5)
optimizer = torch.optim.Adam(simclr.parameters(), lr=3e-4, weight_decay=1e-4)

simclr_losses = []
for epoch in range(EPOCHS):
    simclr.train()
    ep = []
    for x_i, x_j, _ in tqdm(train_loader, desc=f'SimCLR {epoch+1}/{EPOCHS}'):
        x_i, x_j = x_i.to(device), x_j.to(device)
        z_i, z_j, _, _ = simclr(x_i, x_j)
        loss = criterion(z_i, z_j)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        ep.append(loss.item())
    simclr_losses.append(np.mean(ep))
    print(f'Epoch {epoch+1:02d} | Loss: {np.mean(ep):.4f}')

torch.save(simclr.state_dict(), 'saved/simclr.pt')

plt.figure(figsize=(8,3))
plt.plot(simclr_losses, marker='o')
plt.title('SimCLR Training Loss'); plt.xlabel('Epoch'); plt.ylabel('NT-Xent Loss')
plt.grid(True); plt.show()

## SimCLR: Linear Evaluation

Freeze the encoder completely → train only a `nn.Linear(512, 10)` on top using CIFAR-10 labels.

**Why this is the standard benchmark:** A linear layer can only separate *linearly separable* features. High accuracy means the SSL encoder has already organized the feature space semantically — no fine-tuning needed, just reading off the structure that was already there.

> Random baseline: ~10% (10 classes). SimCLR (ResNet-50, full training) reaches ~70% on CIFAR-10. Our smaller setup will be lower, but the gap from random shows learning happened.

In [ ]:
simclr.load_state_dict(torch.load('saved/simclr.pt', map_location=device))
for p in simclr.encoder.parameters(): p.requires_grad = False

clf = nn.Linear(512, 10).to(device)
eval_tf = transforms.Compose([transforms.ToTensor(),
    transforms.Normalize([0.4914,0.4822,0.4465],[0.2023,0.1994,0.2010])])
train_lbl = torchvision.datasets.CIFAR10('./data', train=True,  download=True, transform=eval_tf)
test_lbl  = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=eval_tf)
trl = DataLoader(train_lbl, batch_size=256, shuffle=True,  num_workers=2)
tel = DataLoader(test_lbl,  batch_size=256, shuffle=False, num_workers=2)

opt_clf = torch.optim.Adam(clf.parameters(), lr=1e-3)
for epoch in range(10):
    clf.train(); correct = total = 0
    for imgs, labels in tqdm(trl, desc=f'Linear Eval {epoch+1}/10'):
        imgs, labels = imgs.to(device), labels.to(device)
        with torch.no_grad(): h = torch.flatten(simclr.encoder(imgs), 1)
        loss = F.cross_entropy(clf(h), labels)
        opt_clf.zero_grad(); loss.backward(); opt_clf.step()
        correct += (clf(h).argmax(1) == labels).sum().item(); total += labels.size(0)
    print(f'  Train Acc: {correct/total*100:.2f}%')

clf.eval(); correct = total = 0
simclr_embeddings, simclr_labels = [], []
with torch.no_grad():
    for imgs, labels in tel:
        imgs, labels = imgs.to(device), labels.to(device)
        h = torch.flatten(simclr.encoder(imgs), 1)
        correct += (clf(h).argmax(1) == labels).sum().item(); total += labels.size(0)
        simclr_embeddings.append(h.cpu()); simclr_labels.append(labels.cpu())
simclr_embeddings = torch.cat(simclr_embeddings)
simclr_labels     = torch.cat(simclr_labels)
print(f'\n✅ SimCLR Linear Eval Test Accuracy: {correct/total*100:.2f}%')

---
## Part 2: DINO — The Highlight

DINO (Caron et al., 2021) solves a problem SimCLR can't: **how do you use ViT for SSL?**

SimCLR needs large batches for negatives. BYOL removed negatives but needed an asymmetric predictor MLP. DINO goes further: no negatives, no predictor, no asymmetry — just a **centering trick** to prevent collapse.

### The Architecture
```
              global crop 1 ──▶ [Student] ──▶ softmax(z_s / τ_s) ──┐
              global crop 2 ──▶ [Student] ──▶ softmax(z_s / τ_s) ──┤
              local  crop 1 ──▶ [Student] ──▶ ...                   ├──▶ Cross-Entropy Loss
              local  crop 2 ──▶ [Student] ──▶ ...                   |
global crop 1 ──────────────▶ [Teacher] ──▶ softmax((z_t - c) / τ_t) ─┘
global crop 2 ──────────────▶ [Teacher] ──▶ softmax((z_t - c) / τ_t)
```

**Key ideas:**
- **Multi-crop**: Teacher sees only global crops (>50%), Student sees both global + local (<50%). The student is forced to predict global context from restricted local views.
- **EMA update**: Teacher = exponential moving average of Student. No gradient flows to Teacher.
- **Centering**: Subtract running mean `c` from teacher output to prevent all outputs collapsing to one mode.
- **Temperature sharpening**: Teacher uses very low temperature (sharp distribution) to give a clear learning signal.

### Why centering prevents collapse
Without centering, the teacher could output the same constant vector for every image (trivial solution). Centering forces the mean across batch to be zero, making constant outputs impossible.

### The "wow" result
DINO on ViT produces attention maps that look like **segmentation masks** — the model learns to separate foreground from background without ever seeing a single segmentation label!

<img src="figures/dino_arch.png" width="500"/>

*DINO: student predicts teacher's output distribution. Teacher is an EMA of the student. Centering subtracts a running mean from teacher logits to prevent collapse.*

In [ ]:
# ─── DINO Multi-Crop Augmentation ────────────────────────────────────────────

class DINOAugmentation:
    """
    Creates:
      - 2 global crops (large, scale 0.4–1.0)
      - n_local local crops (small, scale 0.05–0.4)
    Teacher only sees global crops; student sees all.
    """
    def __init__(self, image_size=32, n_local=4):
        normalize = transforms.Normalize([0.4914,0.4822,0.4465],[0.2023,0.1994,0.2010])
        flip_jitter = [
            transforms.RandomHorizontalFlip(),
            transforms.RandomApply([transforms.ColorJitter(0.4,0.4,0.2,0.1)], p=0.8),
            transforms.RandomGrayscale(p=0.2),
        ]
        self.global_transform = transforms.Compose([
            transforms.RandomResizedCrop(image_size, scale=(0.4, 1.0)),
            *flip_jitter,
            transforms.ToTensor(), normalize
        ])
        self.local_transform = transforms.Compose([
            transforms.RandomResizedCrop(image_size, scale=(0.05, 0.4)),
            *flip_jitter,
            transforms.ToTensor(), normalize
        ])
        self.n_local = n_local

    def __call__(self, img):
        global1 = self.global_transform(img)
        global2 = self.global_transform(img)
        locals_ = [self.local_transform(img) for _ in range(self.n_local)]
        return [global1, global2] + locals_   # teacher uses [0,1]; student uses all


class CIFAR10DINO(Dataset):
    def __init__(self, root='./data', train=True, n_local=4):
        self.dataset = torchvision.datasets.CIFAR10(root=root, train=train, download=True)
        self.augment = DINOAugmentation(n_local=n_local)
    def __len__(self): return len(self.dataset)
    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        return self.augment(img), label

In [ ]:
# ─── DINO: Student & Teacher Networks ────────────────────────────────────────
# We use a small ViT (via timm) as the backbone — the original contribution of DINO.
# !pip install timm
import timm

class DINOHead(nn.Module):
    """
    Projection head on top of ViT.  Output is K-dimensional (prototype space).
    Uses weight normalization on the final layer for stability.
    """
    def __init__(self, in_dim=192, hidden_dim=512, out_dim=256, n_layers=3):
        super().__init__()
        layers = [nn.Linear(in_dim, hidden_dim), nn.GELU()]
        for _ in range(n_layers - 2):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.GELU()]
        layers.append(nn.Linear(hidden_dim, out_dim, bias=False))
        self.mlp = nn.Sequential(*layers)
        # Weight-normalized last layer
        self.last_layer = nn.utils.weight_norm(nn.Linear(out_dim, out_dim, bias=False))
        self.last_layer.weight_g.data.fill_(1)

    def forward(self, x):
        x = self.mlp(x)
        x = F.normalize(x, dim=-1, p=2)
        return self.last_layer(x)


def build_dino_model(out_dim=256):
    """Build a small ViT-Tiny backbone + DINO head."""
    vit = timm.create_model('vit_tiny_patch16_224', pretrained=False,
                             img_size=32, patch_size=4, num_classes=0)  # num_classes=0 → no FC
    embed_dim = vit.embed_dim  # 192 for ViT-Tiny
    head = DINOHead(in_dim=embed_dim, out_dim=out_dim)
    return vit, head


# Build student and teacher (identical architecture, separate weights)
student_vit, student_head = build_dino_model()
teacher_vit, teacher_head = build_dino_model()

student_vit, student_head = student_vit.to(device), student_head.to(device)
teacher_vit, teacher_head = teacher_vit.to(device), teacher_head.to(device)

# Teacher starts with same weights as student
teacher_vit.load_state_dict(student_vit.state_dict())
teacher_head.load_state_dict(student_head.state_dict())

# Teacher is NEVER updated via gradients
for p in teacher_vit.parameters():  p.requires_grad = False
for p in teacher_head.parameters(): p.requires_grad = False

total = sum(p.numel() for p in student_vit.parameters()) + sum(p.numel() for p in student_head.parameters())
print(f'Student parameters: {total:,}')

In [ ]:
# ─── DINO Loss ────────────────────────────────────────────────────────────────

class DINOLoss(nn.Module):
    """
    Cross-entropy between softened teacher distribution and student distribution.
    Teacher: low temperature (sharp = confident signal)
    Student: higher temperature (softer)
    Centering: running mean subtracted from teacher logits to prevent collapse.
    """
    def __init__(self, out_dim=256, n_crops=6, warmup_teacher_temp=0.04,
                 teacher_temp=0.04, student_temp=0.1, center_momentum=0.9):
        super().__init__()
        self.student_temp = student_temp
        self.teacher_temp = teacher_temp
        self.n_crops = n_crops
        self.center_momentum = center_momentum
        self.register_buffer('center', torch.zeros(1, out_dim))

    def forward(self, student_out, teacher_out):
        """
        student_out: list of n_crops logits, each (B, out_dim)
        teacher_out: list of 2 logits (global crops only), each (B, out_dim)
        """
        student_out = [s / self.student_temp for s in student_out]
        # Teacher: center + sharpen
        teacher_out = [(t - self.center) / self.teacher_temp for t in teacher_out]
        teacher_probs = [F.softmax(t, dim=-1).detach() for t in teacher_out]

        total_loss = 0
        n_loss_terms = 0
        for t_prob in teacher_probs:
            for s_idx, s_logit in enumerate(student_out):
                # Skip when student and teacher see the same crop
                if s_idx < 2 and s_idx == teacher_probs.index(t_prob): continue
                loss = -( t_prob * F.log_softmax(s_logit, dim=-1) ).sum(dim=-1).mean()
                total_loss += loss
                n_loss_terms += 1

        total_loss /= n_loss_terms
        self.update_center(torch.cat(teacher_out))
        return total_loss

    @torch.no_grad()
    def update_center(self, teacher_output):
        """Running mean update for the centering vector."""
        batch_center = teacher_output.mean(dim=0, keepdim=True)
        self.center = self.center * self.center_momentum + batch_center * (1 - self.center_momentum)

In [ ]:
# ─── DINO Training ───────────────────────────────────────────────────────────

N_LOCAL   = 4      # 2 global + 4 local crops per image
OUT_DIM   = 256
EPOCHS_D  = 10
BATCH_D   = 64
EMA_M     = 0.996  # EMA momentum for teacher update

dino_dataset = CIFAR10DINO(n_local=N_LOCAL)

def dino_collate(batch):
    crops_list, labels = zip(*batch)
    n_views = len(crops_list[0])
    stacked = [torch.stack([crops_list[i][v] for i in range(len(crops_list))]) for v in range(n_views)]
    return stacked, torch.tensor(labels)

dino_loader = DataLoader(dino_dataset, batch_size=BATCH_D, shuffle=True,
                          num_workers=2, drop_last=True, collate_fn=dino_collate)

dino_loss_fn = DINOLoss(out_dim=OUT_DIM, n_crops=2+N_LOCAL).to(device)
optimizer_d  = torch.optim.AdamW(
    list(student_vit.parameters()) + list(student_head.parameters()),
    lr=5e-4, weight_decay=0.04
)

dino_losses = []

for epoch in range(EPOCHS_D):
    student_vit.train(); student_head.train()
    ep = []

    for crops, _ in tqdm(dino_loader, desc=f'DINO {epoch+1}/{EPOCHS_D}'):
        crops = [c.to(device) for c in crops]  # crops[0], crops[1] = global; rest = local

        # ── Student forward: all crops ──
        student_out = [student_head(student_vit(c)) for c in crops]

        # ── Teacher forward: global crops only (no grad) ──
        with torch.no_grad():
            teacher_out = [teacher_head(teacher_vit(crops[0])),
                           teacher_head(teacher_vit(crops[1]))]

        loss = dino_loss_fn(student_out, teacher_out)
        optimizer_d.zero_grad(); loss.backward(); optimizer_d.step()

        # ── EMA update teacher ──
        with torch.no_grad():
            for s_param, t_param in zip(student_vit.parameters(), teacher_vit.parameters()):
                t_param.data = EMA_M * t_param.data + (1 - EMA_M) * s_param.data
            for s_param, t_param in zip(student_head.parameters(), teacher_head.parameters()):
                t_param.data = EMA_M * t_param.data + (1 - EMA_M) * s_param.data

        ep.append(loss.item())

    dino_losses.append(np.mean(ep))
    print(f'Epoch {epoch+1:02d} | Loss: {np.mean(ep):.4f} | Center norm: {dino_loss_fn.center.norm().item():.4f}')

torch.save({'student_vit': student_vit.state_dict(),
            'student_head': student_head.state_dict()}, 'saved/dino.pt')

plt.figure(figsize=(8,3))
plt.plot(dino_losses, marker='o', color='darkorange')
plt.title('DINO Training Loss'); plt.xlabel('Epoch'); plt.ylabel('Cross-Entropy')
plt.grid(True); plt.show()

## DINO: Linear Evaluation

Freeze the **teacher** encoder (most stable representations) and train a linear classifier on top.

> This is the standard SSL benchmark: if a *single linear layer* achieves high accuracy on frozen features, the encoder has learned linearly separable, semantically meaningful representations — without ever seeing a label.

**Expected result:** DINO (ViT backbone) should outperform SimCLR (ResNet backbone) on linear eval, because ViT's patch-based attention captures richer global context.

In [ ]:
ckpt = torch.load('saved/dino.pt', map_location=device)
student_vit.load_state_dict(ckpt['student_vit'])
for p in student_vit.parameters(): p.requires_grad = False

embed_dim = student_vit.embed_dim
clf_dino  = nn.Linear(embed_dim, 10).to(device)
opt_dino_clf = torch.optim.Adam(clf_dino.parameters(), lr=1e-3)

for epoch in range(10):
    clf_dino.train(); correct = total = 0
    for imgs, labels in tqdm(trl, desc=f'DINO Linear Eval {epoch+1}/10'):
        imgs, labels = imgs.to(device), labels.to(device)
        with torch.no_grad(): h = student_vit(imgs)
        loss = F.cross_entropy(clf_dino(h), labels)
        opt_dino_clf.zero_grad(); loss.backward(); opt_dino_clf.step()
        correct += (clf_dino(h).argmax(1)==labels).sum().item(); total += labels.size(0)
    print(f'  Train Acc: {correct/total*100:.2f}%')

clf_dino.eval(); correct = total = 0
dino_embeddings, dino_labels = [], []
with torch.no_grad():
    for imgs, labels in tel:
        imgs, labels = imgs.to(device), labels.to(device)
        h = student_vit(imgs)
        correct += (clf_dino(h).argmax(1)==labels).sum().item(); total += labels.size(0)
        dino_embeddings.append(h.cpu()); dino_labels.append(labels.cpu())
dino_embeddings = torch.cat(dino_embeddings)
dino_labels     = torch.cat(dino_labels)
print(f'\n✅ DINO Linear Eval Test Accuracy: {correct/total*100:.2f}%')

---
## The "Wow" Moment: DINO Attention Maps

This is the result that made DINO famous. The ViT's `[CLS]` token attends to the most semantically relevant patches in an image. When we visualize the attention weights of the **last transformer layer**, we see that the model has learned to **segment objects** without ever seeing a single segmentation label.

> The paper (Caron et al., 2021) visualized this on ImageNet — objects are cleanly separated from background purely from SSL training.

**What to look for:**
- Attention concentrates on the foreground object — not the background
- Different heads may focus on different parts (body, edges, context)
- On CIFAR-10 (32×32), the effect is less dramatic than ImageNet — but the foreground bias should still appear
- A model trained with random weights shows uniform attention — comparing them makes the learned structure obvious

In [ ]:
student_vit.eval()

classes = ['airplane','automobile','bird','cat','deer',
           'dog','frog','horse','ship','truck']
mean = torch.tensor([0.4914,0.4822,0.4465]).view(3,1,1)
std  = torch.tensor([0.2023,0.1994,0.2010]).view(3,1,1)

# Hook to capture attention weights from the last transformer block
attentions = {}
def hook_fn(module, input, output):
    attentions['last'] = output

# Register hook on last attention block
student_vit.blocks[-1].attn.attn_drop.register_forward_hook(hook_fn)

# Get test images
raw_test = torchvision.datasets.CIFAR10('./data', train=False, transform=eval_tf)
img_loader = DataLoader(raw_test, batch_size=1, shuffle=True)

n_patch  = (32 // 4) ** 2   # 64 patches for 32x32 image with patch_size=4
n_heads  = student_vit.blocks[-1].attn.num_heads
patch_h  = patch_w = 32 // 4   # 8x8 grid of patches

fig, axes = plt.subplots(5, n_heads + 1, figsize=(2*(n_heads+1), 12))

sample_iter = iter(img_loader)
for row in range(5):
    img_tensor, label = next(sample_iter)
    img_tensor = img_tensor.to(device)

    with torch.no_grad():
        _ = student_vit(img_tensor)

    # attention shape: (1, n_heads, n_tokens, n_tokens)
    # tokens: [CLS] + 64 patches = 65
    attn = attentions['last']   # (1, n_heads, 65, 65)
    # CLS token attends to all patch tokens: row 0, columns 1:
    cls_attn = attn[0, :, 0, 1:]  # (n_heads, 64)

    # Show original image
    img_disp = torch.clamp(img_tensor[0].cpu() * std + mean, 0, 1).permute(1,2,0).numpy()
    axes[row][0].imshow(img_disp)
    axes[row][0].set_title(f'{classes[label.item()]}', fontsize=9)
    axes[row][0].axis('off')

    # Show each attention head
    for h in range(n_heads):
        head_map = cls_attn[h].reshape(patch_h, patch_w).cpu().numpy()
        head_map = (head_map - head_map.min()) / (head_map.max() - head_map.min() + 1e-8)
        # Upsample to 32x32
        from PIL import Image
        head_up = np.array(Image.fromarray((head_map * 255).astype(np.uint8)).resize((32,32)))
        axes[row][h+1].imshow(img_disp, alpha=0.4)
        axes[row][h+1].imshow(head_up, cmap='hot', alpha=0.7, vmin=0, vmax=255)
        if row == 0: axes[row][h+1].set_title(f'Head {h+1}', fontsize=8)
        axes[row][h+1].axis('off')

col_labels = ['Original'] + [f'Head {h+1}' for h in range(n_heads)]
plt.suptitle('DINO Self-Attention Maps: [CLS] token → patches\n'
             'Each head specializes in different parts of the object — with no segmentation labels!',
             fontsize=11, y=1.01)
plt.tight_layout(); plt.show()

---
## Comparison: SimCLR vs DINO — t-SNE

t-SNE projects the 512-dim / embed-dim feature space down to 2D. If SSL worked, images of the same class should cluster together — even though the model never saw class labels during pretraining.

**What to look for:**
- Well-separated, tight clusters → the encoder captured semantic content
- Mixed, overlapping clusters → the encoder learned augmentation invariance but not semantics
- Compare SimCLR (ResNet) vs DINO (ViT) — expect DINO to show cleaner separation

In [ ]:
from sklearn.manifold import TSNE

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = plt.cm.tab10(np.linspace(0, 1, 10))

for ax, (name, emb, lbls) in zip(axes, [
    ('SimCLR (ResNet-18)', simclr_embeddings, simclr_labels),
    ('DINO (ViT-Tiny)',    dino_embeddings,   dino_labels)
]):
    idx = np.random.choice(len(emb), 2000, replace=False)
    proj = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(emb[idx].numpy())
    for c in range(10):
        mask = lbls[idx].numpy() == c
        ax.scatter(proj[mask,0], proj[mask,1], c=[colors[c]], label=classes[c], alpha=0.6, s=10)
    ax.set_title(f'{name}', fontsize=12)
    ax.legend(fontsize=7, markerscale=2)
    ax.axis('off')

plt.suptitle('t-SNE: Learned Representations on CIFAR-10 (no labels used in training)', fontsize=13)
plt.tight_layout(); plt.show()

# Exercises

1. Centering — DINO's Collapse Prevention
    The centering trick is what separates DINO from BYOL.

    a) In `DINOLoss`, remove the centering by commenting out the line `t_param.data - self.center`. Retrain for 5 epochs. What happens to the loss and the attention maps?

    b) Track the `center` vector norm (`dino_loss_fn.center.norm()`) across training epochs with centering enabled. Plot it. Does it grow, shrink, or stabilize?

    c) Explain in your own words: why does a constant-output teacher cause collapse, and how does centering mathematically prevent this?

2. Multi-Crop — Does It Help?
    DINO's multi-crop strategy forces the student to match global context from local patches.

    a) Train two versions:
    - **Standard**: 2 global + 4 local crops (default)
    - **No local crops**: 2 global crops only (`n_local=0`)

    b) Compare linear evaluation accuracy on both:

    | Setting | Linear Eval Accuracy |
    |---|---|
    | 2 global + 4 local crops | ? |
    | 2 global crops only | ? |

    c) Visualize attention maps from both models. Do the maps look more focused with or without local crops?

3. Attention Head Specialization

    Different attention heads in DINO tend to specialize (e.g., one focuses on the subject, another on the background).

    a) For 10 different images (one per CIFAR-10 class), visualize all attention heads side by side.

    b) Across your 10 images, do any heads consistently focus on the same types of features (edges, foreground, background)? Describe what you observe.

    c) Compute the **cosine similarity between attention maps of different heads** for the same image. Are some heads more similar to each other than others?

4. (Challenge): SimCLR vs DINO — Full Comparison

    a) Fill in the comparison table:

    | Metric | SimCLR | DINO |
    |---|---|---|
    | Backbone | ResNet-18 | ViT-Tiny |
    | Needs negative pairs? | Yes | No |
    | Linear Eval Accuracy | ? | ? |
    | Training time per epoch | ? | ? |
    | t-SNE cluster quality (subjective 1-5) | ? | ? |
    | Has interpretable attention maps? | No | Yes |

    b) Based on what you've seen in the attention maps, explain *why* DINO would naturally produce better segmentation features than SimCLR.

    c) MAE (masked autoencoders) has become more popular than DINO for general pre-training. List two reasons why MAE won out overall, and one reason why DINO is still preferred for CV-only tasks.

## Submission

Submit your work to GitHub. Your repository should contain:

### 1. Training Script (`run.py`)

```bash
# Train SimCLR
python3 run.py --model simclr --dataset cifar10 --epochs 10 --train

# Train DINO
python3 run.py --model dino   --dataset cifar10 --epochs 10 --train

# Linear evaluation (frozen encoder)
python3 run.py --model simclr --weights saved/simclr.pt --evaluate --linear
python3 run.py --model dino   --weights saved/dino.pt   --evaluate --linear

# Ablation: DINO without centering
python3 run.py --model dino --no-centering --epochs 5 --train

# Ablation: DINO without local crops
python3 run.py --model dino --n-local 0 --epochs 10 --train
```

### 2. `README.md`

Your `README.md` must include:

**Commands used** (exact commands you ran)

**Results table:**

| Model | Linear Eval Acc | Time/epoch | Notes |
|---|---|---|---|
| SimCLR (ResNet-18) | ? | ? | contrastive baseline |
| DINO (ViT-Tiny) | ? | ? | self-distillation |
| DINO (no centering) | ? | ? | collapse ablation |
| DINO (no local crops) | ? | ? | multi-crop ablation |

**Visualizations** (include in README or as separate image files):
- Loss curves for SimCLR and DINO
- Attention map grid (Exercise 3, 10 images × all heads)
- t-SNE comparison: SimCLR vs DINO

**Discussion** (3–5 sentences): Which approach would you use for a medical image segmentation project, and why?